## Generalized Assignment Problem

In a GAP model, we have a set of “machines” M = {1,2,…,m} and a set of “jobs” N = {1,2,…,n} that must be assigned to machines. Each machine i has a capacity $b_i$ units of work. Each job j requires $a_{ij}$ units of work to be completed if it is scheduled on machine i. All jobs must be assigned to exactly one machine. There is a fixed cost $h_i$ of using machine i.


In [23]:
using JuMP, Gurobi

M = [:1 :2 :3 :4]
N = [:1 :2 :3 :4 :5]

b = zeros(4)
h = zeros(4)
for i in 1:4
    b[i] = round(20*rand() + 5,2)
    h[i] = round(10*rand() + 5,2)
end

a = zeros(4,5)
c = zeros(4,5)
for i in 1:4
    for j in 1:5
        a[i,j] = round(10*rand(),2)
        c[i,j] = round(5*rand(),2)
    end
end


m = Model(solver=GurobiSolver(OutputFlag=0))

@variable(m, x[M,N], Bin)
@variable(m, z[M], Bin)

@objective(m, Min, sum(c[i,j]*x[i,j] for i in M for j in N)
            + sum(h[i]*z[i] for i in M))
                
@constraint(m, capacity[i in M], sum(a[i,j]*x[i,j] for j in N) <= b[i])
@constraint(m, jobassign[i in N], sum(x[j,i] for j in M) == 1)

# # Fixed cost logic: option 1
# @constraint(m, logic[i in M, j in N], x[i,j] <= 1*z[i])
                
# # Fixed cost logic: option 2
# @constraint(m, logic[i in M], sum(x[i,j] for j in N) <= length(N)*z[i])

# Fixed cost logic: option 3
@constraint(m, logic[i in M], sum(a[i,j]*x[i,j] for j in N) <= b[i]*z[i])
                
@time(solve(m))

Academic license - for non-commercial use only
  0.010649 seconds (127 allocations: 22.578 KiB)


:Optimal